# Puzzle Similarity Demo - PyTorch and Integrated CUDA Backend

This notebook demonstrates PuzzleSim with the reference PyTorch backend and the integrated automatic CUDA backend, allowing for direct comparison of functionality and performance.

In [ ]:
import torch
import numpy as np
import utils
import time
import sys
import os
from pathlib import Path

# Environment setup
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Import implementations
from puzzle_sim import PuzzleSim as PuzzleSimPyTorch

cuda_available = torch.cuda.is_available()
if cuda_available:
    print("✅ CUDA backend available through puzzle_sim")
else:
    print("❌ CUDA backend not available - CUDA device not detected")

# Configuration
COMPARE_IMPLEMENTATIONS = cuda_available
print(f"🔄 Implementation comparison: {'Enabled' if COMPARE_IMPLEMENTATIONS else 'Disabled'}")
print(f"🖥️  CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available'}")

In [ ]:
# Configuration
dataset = "flowers"  # select one of: "flowers", "garden", "playroom"
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
resize_dims = (416, 628)

# Load data
print(f"📁 Loading dataset: {dataset}")
priors, test_images, test_image_names = utils.load_images(dataset, device)
print(f"Loaded {len(test_images)} test images and {len(priors)} priors.")

# Initialize implementations
implementations = {}

# PyTorch implementation
print("🔧 Initializing PyTorch implementation...")
start_time = time.time()
puzzle_pytorch = PuzzleSimPyTorch(priors, resize=resize_dims, backend='torch')
pytorch_init_time = time.time() - start_time
implementations['PyTorch'] = {
    'model': puzzle_pytorch,
    'init_time': pytorch_init_time,
    'device': 'CPU/GPU'
}
print(f"   Initialization time: {pytorch_init_time:.3f}s")

# CUDA backend (if available)
if COMPARE_IMPLEMENTATIONS:
    print("🚀 Initializing CUDA backend...")
    try:
        start_time = time.time()
        puzzle_cuda = PuzzleSimPyTorch(priors, resize=resize_dims, backend='auto')
        cuda_init_time = time.time() - start_time
        implementations['CUDA'] = {
            'model': puzzle_cuda,
            'init_time': cuda_init_time,
            'device': 'CUDA'
        }
        print(f"   Initialization time: {cuda_init_time:.3f}s")
        print(f"   🏆 CUDA speedup in init: {pytorch_init_time/cuda_init_time:.2f}x")
        COMPARE_IMPLEMENTATIONS = True
    except RuntimeError as e:
        print(f"   ❌ CUDA backend initialization failed: {e}")
        print("   🔧 Optional compiled-kernel build:")
        print("      - Run from the repository root")
        print("      - Then: PUZZLE_SIM_BUILD_CUDA=1 pip install --no-build-isolation -v .")
        print("      - Or use the pure PyTorch fallback without building the optional extension")
        COMPARE_IMPLEMENTATIONS = False
        cuda_available = False

print(f"\n📊 Available implementations: {list(implementations.keys())}")

In [ ]:
utils.plot_image_tensor_row(test_images, test_image_names)

In [ ]:
# select one of the test images
selected = "00007.png"
test_image = test_images[np.argwhere(np.array(test_image_names) == selected)[0][0]]

print(f"🖼️  Selected test image: {selected}")
print(f"    Shape: {test_image.shape}")
print(f"    Device: {test_image.device}")

# Run inference with all available implementations
results = {}

for impl_name, impl_info in implementations.items():
    print(f"\n🔄 Running {impl_name} implementation...")
    model = impl_info['model']
    
    # Warm up (for fair timing)
    with torch.no_grad():
        _ = model(test_image)
    
    # Timed run
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start_time = time.perf_counter()
    
    with torch.no_grad():
        similarity_map = model(test_image)
    
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    inference_time = time.perf_counter() - start_time
    
    results[impl_name] = {
        'similarity_map': similarity_map,
        'inference_time': inference_time,
        'shape': similarity_map.shape,
        'range': (similarity_map.min().item(), similarity_map.max().item()),
        'device': similarity_map.device
    }
    
    print(f"    ⏱️  Inference time: {inference_time*1000:.1f}ms")
    print(f"    📐 Output shape: {similarity_map.shape}")
    print(f"    📊 Value range: [{similarity_map.min():.4f}, {similarity_map.max():.4f}]")

# Performance comparison
if len(results) > 1:
    pytorch_time = results['PyTorch']['inference_time']
    cuda_time = results['CUDA']['inference_time']
    speedup = pytorch_time / cuda_time
    print(f"\n🏆 Performance Summary:")
    print(f"    PyTorch: {pytorch_time*1000:.1f}ms")
    print(f"    CUDA:    {cuda_time*1000:.1f}ms")
    print(f"    Speedup: {speedup:.2f}x")

In [ ]:
# Visualization
import matplotlib.pyplot as plt

if len(results) == 1:
    # Single implementation visualization
    impl_name = list(results.keys())[0]
    similarity_map = results[impl_name]['similarity_map']
    
    print(f"📊 Visualizing results from {impl_name} implementation")
    utils.plot_image_tensor(test_image)
    utils.plot_heatmap_tensor(similarity_map)
    
else:
    # Comparison visualization
    fig, axes = plt.subplots(2, len(results) + 1, figsize=(5 * (len(results) + 1), 10))
    
    # Show test image in first column
    img_np = test_image.squeeze().cpu().permute(1, 2, 0).numpy()
    axes[0, 0].imshow(img_np)
    axes[0, 0].set_title("Test Image")
    axes[0, 0].axis('off')
    axes[1, 0].axis('off')  # Empty space below test image
    
    # Show results from each implementation
    for i, (impl_name, result) in enumerate(results.items()):
        similarity_map = result['similarity_map'].cpu().numpy()
        inference_time = result['inference_time']
        
        # Top row: similarity maps
        im = axes[0, i + 1].imshow(similarity_map, cmap='jet')
        axes[0, i + 1].set_title(f'{impl_name}\n{inference_time*1000:.1f}ms')
        axes[0, i + 1].axis('off')
        plt.colorbar(im, ax=axes[0, i + 1], fraction=0.046, pad=0.04)
        
        # Bottom row: difference visualization (if more than one implementation)
        if len(results) > 1 and i > 0:
            # Compare with first implementation
            first_result = list(results.values())[0]['similarity_map'].cpu().numpy()
            diff = np.abs(similarity_map - first_result)
            
            im_diff = axes[1, i + 1].imshow(diff, cmap='hot')
            axes[1, i + 1].set_title(f'|{impl_name} - {list(results.keys())[0]}|\nMax diff: {diff.max():.4f}')
            axes[1, i + 1].axis('off')
            plt.colorbar(im_diff, ax=axes[1, i + 1], fraction=0.046, pad=0.04)
        else:
            axes[1, i + 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Accuracy comparison
    if len(results) > 1:
        impl_names = list(results.keys())
        map1 = results[impl_names[0]]['similarity_map']
        map2 = results[impl_names[1]]['similarity_map']
        
        max_diff = torch.max(torch.abs(map1 - map2)).item()
        relative_error = max_diff / torch.max(torch.abs(map1)).item()
        
        print(f"\n🔍 Accuracy Comparison:")
        print(f"    Maximum absolute difference: {max_diff:.6f}")
        print(f"    Relative error: {relative_error:.6f}")
        
        if relative_error < 1e-3:
            print("    ✅ Results match within tolerance!")
        else:
            print("    ⚠️  Large difference detected!")

In [ ]:
# Advanced Benchmarking and Configuration Testing

if COMPARE_IMPLEMENTATIONS and 'CUDA' in implementations:
    print("🚀 Advanced CUDA Features Testing")
    print("=" * 50)
    
    # Test different memory modes (CUDA backend feature)
    memory_modes = [
        {'name': 'Memory-Save (stride=2)', 'mem_save': True, 'stride': 2},
        {'name': 'Memory-Save (stride=4)', 'mem_save': True, 'stride': 4},
        {'name': 'Memory-Save (stride=8)', 'mem_save': True, 'stride': 8},
        {'name': 'Naive Mode', 'mem_save': False, 'stride': 4},
    ]
    
    print("\n💾 Testing different memory modes...")
    cuda_model = implementations['CUDA']['model']
    
    memory_results = {}
    for mode in memory_modes:
        print(f"\nTesting {mode['name']}...")
        try:
            torch.cuda.reset_peak_memory_stats()
            
            start_time = time.perf_counter()
            with torch.no_grad():
                result = cuda_model(test_image, mem_save=mode['mem_save'], stride=mode['stride'])
            exec_time = time.perf_counter() - start_time
            
            peak_memory = torch.cuda.max_memory_allocated() / 1024**2  # MB
            
            memory_results[mode['name']] = {
                'time': exec_time,
                'memory': peak_memory,
                'result': result
            }
            
            print(f"  ⏱️  Time: {exec_time*1000:.1f}ms")
            print(f"  🧠 Memory: {peak_memory:.1f}MB")
            
        except Exception as e:
            print(f"  ❌ Failed: {e}")
    
    # Test different network architectures
    print("\n🧠 Testing different network architectures...")
    net_types = ['squeeze', 'alex', 'vgg']
    
    architecture_results = {}
    for net_type in net_types:
        print(f"\nTesting {net_type} network...")
        try:
            # Create smaller test for speed
            small_resize = (128, 128)
            test_puzzle = PuzzleSimPyTorch(priors, net_type=net_type, resize=small_resize, backend='auto')
            
            start_time = time.perf_counter()
            with torch.no_grad():
                result = test_puzzle(test_image)
            exec_time = time.perf_counter() - start_time
            
            architecture_results[net_type] = {
                'time': exec_time,
                'result_shape': result.shape,
                'result_range': (result.min().item(), result.max().item())
            }
            
            print(f"  ⏱️  Time: {exec_time*1000:.1f}ms")
            print(f"  📐 Shape: {result.shape}")
            print(f"  📊 Range: [{result.min():.3f}, {result.max():.3f}]")
            
        except Exception as e:
            print(f"  ❌ {net_type} failed: {e}")
    
    print("\n📊 Architecture Performance Summary:")
    for net_type, result in architecture_results.items():
        print(f"    {net_type:>8}: {result['time']*1000:>6.1f}ms")

else:
    print("🔧 CUDA backend not available - skipping advanced features testing")
    print("Use CUDA tensors to enable CUDA backend benchmarks")


In [ ]:
# Summary and Recommendations

print("🎯 Performance Summary & Recommendations")
print("=" * 60)

if COMPARE_IMPLEMENTATIONS and 'CUDA' in results and 'PyTorch' in results:
    # Overall performance comparison
    pytorch_time = results['PyTorch']['inference_time']
    cuda_time = results['CUDA']['inference_time']
    total_speedup = pytorch_time / cuda_time
    
    print(f"📈 Overall Performance Comparison:")
    print(f"    PyTorch Implementation:  {pytorch_time*1000:>8.1f}ms")
    print(f"    CUDA Implementation:     {cuda_time*1000:>8.1f}ms") 
    print(f"    🚀 Total Speedup:        {total_speedup:>8.2f}x")
    
    # Performance rating
    if total_speedup > 5:
        print("    🏆 Excellent performance improvement!")
        recommendation = "Use CUDA backend for production workloads"
    elif total_speedup > 2:
        print("    ✅ Good performance improvement")
        recommendation = "CUDA backend recommended for most use cases"
    else:
        print("    ⚠️  Modest improvement")
        recommendation = "Consider PyTorch for simplicity, CUDA for larger datasets"
    
    # Memory efficiency (if tested)
    if 'memory_results' in locals() and memory_results:
        print(f"\n💾 Memory Efficiency:")
        best_memory = min(memory_results.values(), key=lambda x: x['memory'])
        best_time = min(memory_results.values(), key=lambda x: x['time'])
        
        for mode_name in memory_results:
            result = memory_results[mode_name]
            marker = "🏆" if result == best_memory else "⚡" if result == best_time else "  "
            print(f"    {marker} {mode_name:<25}: {result['time']*1000:>6.1f}ms, {result['memory']:>6.1f}MB")
    
    print(f"\n💡 Recommendation: {recommendation}")
    
else:
    print("📝 Single Implementation Results:")
    impl_name = list(results.keys())[0] 
    inference_time = results[impl_name]['inference_time']
    print(f"    Implementation: {impl_name}")
    print(f"    Inference Time: {inference_time*1000:.1f}ms")
    print(f"    Status: {'✅ Working correctly' if len(results) > 0 else '❌ Issues detected'}")
    
    if not cuda_available:
        print(f"\n💡 To enable CUDA acceleration and comparison:")
        print(f"    1. Stay in the repository root")
        print(f"    2. Build optional extension: PUZZLE_SIM_BUILD_CUDA=1 pip install --no-build-isolation -v .")
        print(f"    3. Restart notebook and re-run cells")

print(f"\n🔧 System Information:")
print(f"    PyTorch Version: {torch.__version__}")
print(f"    CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"    CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"    CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")

print(f"\n✨ Demo completed successfully!")
print(f"Dataset: {dataset} | Test Image: {selected}")
print(f"Implementations tested: {list(implementations.keys())}")
